# Espinosa et al. (2017) on CRCNS bf-2

Paper: Espinosa N, Alonso A, Morales C, Espinosa P, Chávez AE, Fuentealba P (2017). Basal Forebrain
Gating by Somatostatin Neurons Drives Prefrontal Cortical Activity. Cereb Cortex 29:42-53,
doi:10.1093/cercor/bhx302.

Dataset: CRCNS bf-2, "Basal forebrain extracellular recordings of anesthetized mice with somatostatin
positive neurons optogenetically inhibited", doi:10.6080/K0MG7MQW, read from the public bucket
`crcnsarchive` under `bf-2/live/`. CRCNS data sets are not versioned, so the version is recorded as
`unversioned` (docs/archives.md, CRCNS); the paper cites no version, and there is only the one
published copy, so it is the one used. The files read and their MD5 sums from the data set's
`checksums.md5` (read 2026-11-09) are printed below, so a later change can be detected. The data
descriptor is version 0.6 (Dec 12, 2018).

Claims: `claims/espinosa-2017.yaml`, written from the version of record with its supplement and
checked by a second reader.

| Claim | Tested here |
|---|---|
| supp1-nphr-in-somatostatin-cells | Blocked: data missing (histology) |
| supp2-nphr-silences-som-cells | Blocked: data missing (in vitro) |
| supp3-two-ventral-pallidum-cell-types | Blocked: data missing (in vitro) |
| fig1-bf-response-classes | Tested |
| fig1d-excited-cells-fire-faster | Tested |
| fig1d-discharge-by-class | Tested |
| fig1e-excited-slower-kinetics | Tested |
| supp4-tail-pinch-response-patterns | Tested |
| fig2c, supp6, fig2d, supp7c, fig2e, fig3a, fig3b, supp9 (3 claims) | Blocked: data missing (prefrontal units) |
| fig4bcd, supp10, fig4g, supp11a, fig4h, suppt2 | Blocked: data missing (prefrontal LFP) |
| fig5c-light-path-needed | Blocked: data missing (light-blocked sessions) |
| fig5d-nphr-locomotion, supp14-moving-no-difference | Blocked: metadata insufficient (no laser timing for NpHR- videos) |

The data set holds the ventral pallidum (basal forebrain) optrode recordings of Figure 1 and
Supplementary Figure 4, medial septum recordings for a later paper, and open-field videos. It holds
no prefrontal recordings, no in vitro or histology data, and no sessions with the light path blocked.

In [ ]:
import collections
import resource
import time
from concurrent.futures import ProcessPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

import bf2

T0 = time.time()
for k, v in bf2.MD5.items():
    print(f"{v}  {k}")
print("dataset:", bf2.DATASET)

## Loading

Each archive is a tar.gz file, streamed once from start to end; only the unit tables
(`data_correlation.mat`) and the light and pinch timing of each recording (`recxx_lan500.mat`, without
its LFP array) are kept. The two open-field archives are streamed for their file lists and small
.mat files. The six streams run in parallel.

In [ ]:
jobs = [("ephys", a) for a in bf2.VP_ANIMALS] + [
    ("beh", "Behavioral_tests/OPEN_FIELD/VP.tar.gz"), ("beh", "Behavioral_tests/OPEN_FIELD/Control.tar.gz")]


def load(job):
    kind, what = job
    return bf2.stream_animal(what) if kind == "ephys" else bf2.stream_behavior(what)


with ProcessPoolExecutor(6) as ex:
    loaded = list(ex.map(load, jobs))
animals = loaded[:4]
beh_vp, beh_ctl = loaded[4], loaded[5]
print(f"loaded in {time.time() - T0:.0f} s")

## Data check

In [ ]:
units, unit_animal = [], []
lan = {}
for r in animals:
    for rn, rec in r["recs"].items():
        lan[(r["animal"], rn.replace("rec", "reg"))] = rec["lan"]
    for u in r["units"]:
        units.append(u)
        unit_animal.append(r["animal"])
unit_animal = np.array(unit_animal)
print("header of data_correlation.mat:", animals[0]["header"])
print(f"animals {len(animals)}, recordings {len(lan)}, units {len(units)} (paper: 4 animals, 330 units; "
      "descriptor: 20 recordings, 330 neurons)")
for r in animals:
    cnt = collections.Counter(u["rec"] for u in r["units"])
    print(f"\n{r['animal']}: bad channels {r['bad_channel']}")
    for rn, rec in sorted(r["recs"].items()):
        L = rec["lan"]
        on, off = bf2.onsets_from_ttl(L["ttl"][:, 0], L["srate"])
        line = (f"  {rn} {rec['rhd']}: srate {L['srate']:.0f} Hz, {L['pnts'] / L['srate']:.1f} s, LFP array "
                f"{L['data_shape']}, laser TTL {len(on)} pulses of {np.median(off - on):.3f} s every "
                f"{np.median(np.diff(on)):.2f} s, good-signal fraction {L['selected'].mean():.3f}, "
                f"units {cnt[rn.replace('rec', 'reg')]}")
        if "RTb" in L:
            on2, off2 = bf2.onsets_from_ttl(L["ttl"][:, 1], L["srate"])
            line += (f"; second TTL row {len(on2)} pulses (median {np.median(off2 - on2):.2f} s); "
                     f"LAN.RTa {len(L['RTa'])}, LAN.RTb {len(L['RTb'])} events")
        print(line)

In [ ]:
# Per unit: laser intensities, the stored rate against the rate counted from the spike times, the
# per-unit laser and pinch onset lists, and the dataset's own response flags.
print("intensity (column 'intensity'):", collections.Counter(u["intensity"] for u in units))
print("'Int. real':", collections.Counter(round(u["int_real"], 1) for u in units))
print("pulse durations (s):", collections.Counter(tuple(np.round(u["pulse"], 2)) for u in units))
dur = np.array([lan[(a, u["rec"])]["pnts"] / lan[(a, u["rec"])]["srate"] for u, a in zip(units, unit_animal)])
rate_counted = np.array([len(u["spikes"]) for u in units]) / dur
frate = np.array([u["frate"] for u in units])
print(f"spike times in ms: range {min(u['spikes'].min() for u in units):.1f} to "
      f"{max(u['spikes'].max() for u in units):.1f}; recording lengths {sorted(set(np.round(dur)))} s")
print(f"unsorted trains {sum((np.diff(u['spikes']) < 0).any() for u in units)}, "
      f"repeated times {sum((np.diff(u['spikes']) == 0).any() for u in units)}")
print(f"stored 'f rate' / rate counted over the whole recording: median {np.median(frate / rate_counted):.3f}, "
      f"range {np.min(frate / rate_counted):.3f} to {np.max(frate / rate_counted):.3f}")
print(f"spike counts per unit: {min(len(u['spikes']) for u in units)} to {max(len(u['spikes']) for u in units)}; "
      f"units with exactly 20000 spikes: {sum(len(u['spikes']) == 20000 for u in units)}")
sig = collections.Counter((u["spikes"].size, round(u["spikes"][0], 2), round(u["spikes"][-1], 2)) for u in units)
print("spike trains deposited twice (same count, first and last time):", sum(v > 1 for v in sig.values()))

# Per-unit laser onsets against the laser TTL.
worst, n_laser = [], collections.Counter()
for u, a in zip(units, unit_animal):
    L = lan[(a, u["rec"])]
    on, _ = bf2.onsets_from_ttl(L["ttl"][:, 0], L["srate"])
    d = np.abs(on[:, None] * 1000 - u["rt_laser"][None, :]).min(axis=0)
    worst.append(d.max())
    n_laser[(a, len(on), len(u["rt_laser"]), len(u["rt_somatic"]))] += 1
print(f"\nevery per-unit laser onset ('RT laser') is within {max(worst):.1f} ms of a rising edge of the laser TTL")
print("(animal, TTL pulses, 'RT laser' onsets, 'RT somatic' onsets): units")
for k, v in sorted(n_laser.items()):
    print("  ", k, v)
for a in ("NE52", "NE58"):
    r = [x for x in animals if x["animal"] == a][0]
    u = r["units"][0]
    L = lan[(a, u["rec"])]
    near = np.array([np.abs(L["RTb"] - t).min() for t in u["rt_laser"]])
    pin = np.array([np.abs(L["RTa"] - t).min() for t in u["rt_somatic"]])
    print(f"{a} {u['rec']}: 'RT laser' onsets nearest pinch (LAN.RTb) {near.min():.0f} ms or more away; "
          f"'RT somatic' onsets nearest laser (LAN.RTa) {pin.max():.0f} ms or less away")

flags = np.array([u["sig_laser"] for u in units])
print("\ndataset's own light flags (column 17, 1 excited, -1 inhibited, 0 none):",
      {k: int((flags == k).sum()) for k in (1, -1, 0)}, "paper: 56, 29, 245")
for a in bf2.VP_ANIMALS:
    print("  ", a, {k: int(((flags == k) & (unit_animal == a)).sum()) for k in (1, -1, 0)})

In recordings with tail pinch (NE52, NE58) the light pulses alternate between light alone and light
together with the pinch: the per-unit list `RT laser` holds only the light-alone pulses, and
`RT somatic` the pinch-alone ones (printed above). The analysis uses the per-unit lists, so light
responses are measured on light-alone pulses.

**Clock check.** The spikes and the light TTL come from the same Intan file. As a contrast, the
mean absolute log ratio of the rate in the 5 s after each onset to the 5 s before, over units, is
compared between the true onsets and the onsets moved 10 s later (into the light-off period).

In [ ]:
def logratio(sp, onsets):
    a = np.array([((sp >= o) & (sp < o + 5)).sum() for o in onsets]).sum()
    b = np.array([((sp >= o - 5) & (sp < o)).sum() for o in onsets]).sum()
    return np.log2((a + 1) / (b + 1))


rows = collections.defaultdict(lambda: [[], []])
for u, a in zip(units, unit_animal):
    sp, on = u["spikes"] / 1000, u["rt_laser"] / 1000
    rows[(a, u["rec"])][0].append(abs(logratio(sp, on)))
    rows[(a, u["rec"])][1].append(abs(logratio(sp, on + 10)))
n_larger = 0
for k, (t, s) in sorted(rows.items()):
    n_larger += np.mean(t) > np.mean(s)
    print(f"{k[0]} {k[1]}: true onsets {np.mean(t):.3f}, shifted 10 s {np.mean(s):.3f} (units {len(t)})")
print(f"recordings where the response to the true onsets is larger: {n_larger} of {len(rows)}")

In [ ]:
# Open-field archives: members, and which mice have laser timing ('ttl') and a frame-difference vector ('cambio').
for b in (beh_vp, beh_ctl):
    mice = collections.defaultdict(list)
    for name, size in b["members"]:
        if "/segm/" in name:
            continue
        mice[name.split("/")[1]].append((name.split("/")[-1], size))
    print(f"\n{b['path']}: {len(mice)} mice, {len(b['members'])} files")
    for m, files in mice.items():
        mats = {k: bf2.read_mat_any(v) for k, v in b["mats"].items() if f"/{m}/" in k}
        has = {k.split("/")[-1]: sorted(x for x in v if x in ("ttl", "cambio", "laser", "trajectories")) for k, v in mats.items()}
        timing = [k for k, v in has.items() if "ttl" in v]
        ttl_n = [np.size(mats[k2]["ttl"]) for k2 in mats if "ttl" in mats[k2]]
        print(f"  {m}: videos {[f for f, s in files if f.lower().endswith(('.avi', '.mpg'))]}, "
              f"files with laser timing {timing} (trains {ttl_n})")

import io, fsspec, scipy.io as sio
with fsspec.open(f"{bf2.BUCKET}/Behavioral_tests/OPEN_FIELD/result_rapidez_all.mat") as f:
    rap = sio.loadmat(io.BytesIO(f.read()))
for k, v in rap.items():
    if not k.startswith("__"):
        print(f"result_rapidez_all.mat {k}: shape {v.shape}, range {np.nanmin(v):.3g} to {np.nanmax(v):.3g}")

## fig1-bf-response-classes

> Light inhibition of somatostatin cells in vivo changes the firing of a minority of basal forebrain
> units. Of 330 units (4 animals), 29 (8.8%) decrease their activity by 49.2% and 56 (17%) increase it
> by 39.6%; 245 are unaffected.

Method from the claims file (opening comment, shared method): per unit, a 30-s window around each
light onset, 500-ms bins (250-ms variant), normalized by the spikes in the window (n1 per pulse, n2
summed), 100 shuffles of inter-spike and inter-pulse intervals, P < 0.01, readings r1 to r3.
Pulse bins are those whose centres fall in (0, 5] s (bf2.py). Seeds: this section's generator is
seeded per unit from 1 000 000 plus the unit's index.

In [ ]:
t = time.time()
KEYS = [(0.5, "n1"), (0.5, "n2"), (0.25, "n1"), (0.25, "n2")]
cls = []
for i, u in enumerate(units):
    sp, on = u["spikes"] / 1000, u["rt_laser"] / 1000
    cls.append({k: bf2.classify_unit(sp, on, k[0], k[1], seed=1_000_000 + i) for k in KEYS})
print(f"classified {len(units)} units under {len(KEYS)} bin and normalization readings in {time.time() - t:.0f} s")
print(f"units with a pulse window without spikes (n1 drops those pulses): "
      f"{sum(any(((u['spikes'] / 1000 >= o - 15.25) & (u['spikes'] / 1000 < o + 15.25)).sum() == 0 for o in u['rt_laser'] / 1000) for u in units)}")
print(f"units with an undefined normalized curve: {sum(not np.isfinite(c[k]['pulse_mean']) for c in cls for k in KEYS)}")


def classes(key, r):
    return np.array([c[key][r] for c in cls])


def pct(key):
    return np.array([c[key]["pct"] for c in cls])


def within(x, v, tol):
    return abs(x - v) <= tol + 1e-9


rows = []
for key in KEYS:
    for r in ("r1", "r2", "r3"):
        c, p = classes(key, r), pct(key)
        inh, exc, una = (c == -1).sum(), (c == 1).sum(), (c == 0).sum()
        di, de = -np.mean(p[c == -1]), np.mean(p[c == 1])
        counts_ok = within(inh, 29, 2.9) and within(exc, 56, 5.6) and within(una, 245, 24.5)
        change_ok = within(di, 49.2, 5.1) and within(de, 39.6, 7.9)
        rows.append((key, r, inh, exc, una, di, de, counts_ok, change_ok))
        print(f"{key[0] * 1000:.0f} ms {key[1]} {r}: inhibited {inh}, excited {exc}, unaffected {una}; "
              f"decrease {di:.1f}% (s.e.m. {stats.sem(-p[c == -1]):.1f}), increase {de:.1f}% "
              f"(s.e.m. {stats.sem(p[c == 1]):.1f}); counts within tolerance {counts_ok}, changes within {change_ok}")
print("\nexpected: inhibited 29 +- 2.9, excited 56 +- 5.6, unaffected 245 +- 24.5, decrease 49.2 +- 5.1, increase 39.6 +- 7.9")
print(f"across readings: inhibited {min(r[2] for r in rows)} to {max(r[2] for r in rows)}, excited "
      f"{min(r[3] for r in rows)} to {max(r[3] for r in rows)}, unaffected {min(r[4] for r in rows)} to {max(r[4] for r in rows)}, "
      f"decrease {min(r[5] for r in rows):.1f} to {max(r[5] for r in rows):.1f}%, increase {min(r[6] for r in rows):.1f} to {max(r[6] for r in rows):.1f}%")
print("readings with all three counts within tolerance:", sum(r[7] for r in rows), "of", len(rows))
print("readings with both changes within tolerance:", sum(r[8] for r in rows), "of", len(rows))

In [ ]:
# Main reading for the record: 500-ms bins (Methods), n2, r1. By animal:
key = (0.5, "n2")
c = classes(key, "r1")
for a in bf2.VP_ANIMALS:
    m = unit_animal == a
    print(f"{a}: units {m.sum()}, inhibited {(c[m] == -1).sum()}, excited {(c[m] == 1).sum()}, unaffected {(c[m] == 0).sum()}")
# Agreement with the dataset's own flags (column 17), under each reading.
for key in KEYS:
    for r in ("r1", "r2", "r3"):
        cc = classes(key, r)
        print(f"{key[0] * 1000:.0f} ms {key[1]} {r}: same class as the dataset's flag for {(cc == flags).sum()} of {len(cc)} units")
# Variant: the dataset's own classes, with the percent change measured here.
for key in KEYS:
    p = pct(key)
    print(f"variant, dataset's own classes, {key[0] * 1000:.0f} ms {key[1]}: inhibited {(flags == -1).sum()} "
          f"decrease {-np.mean(p[flags == -1]):.1f}% (s.e.m. {stats.sem(-p[flags == -1]):.1f}); excited {(flags == 1).sum()} "
          f"increase {np.mean(p[flags == 1]):.1f}% (s.e.m. {stats.sem(p[flags == 1]):.1f})")

In [ ]:
# Reading on the pulses used: the paper does not say whether light pulses given together with the
# pinch (NE52, NE58) count. Main analysis: the per-unit light-alone onsets ('RT laser'). Here: every
# light pulse of the laser TTL. Seeds: 3 000 000 plus the unit's index.
cls_all = []
for i, (u, a) in enumerate(zip(units, unit_animal)):
    L = lan[(a, u["rec"])]
    on, _ = bf2.onsets_from_ttl(L["ttl"][:, 0], L["srate"])
    cls_all.append({k: bf2.classify_unit(u["spikes"] / 1000, on, k[0], k[1], seed=3_000_000 + i) for k in KEYS})
n_counts = 0
for key in KEYS:
    for r in ("r1", "r2", "r3"):
        c = np.array([x[key][r] for x in cls_all])
        p = np.array([x[key]["pct"] for x in cls_all])
        ok = within((c == -1).sum(), 29, 2.9) and within((c == 1).sum(), 56, 5.6) and within((c == 0).sum(), 245, 24.5)
        n_counts += ok
        print(f"all TTL pulses, {key[0] * 1000:.0f} ms {key[1]} {r}: inhibited {(c == -1).sum()}, excited {(c == 1).sum()}, "
              f"unaffected {(c == 0).sum()}; decrease {-np.mean(p[c == -1]):.1f}%, increase {np.mean(p[c == 1]):.1f}%; counts within tolerance {ok}")
print(f"readings with all three counts within tolerance, all TTL pulses: {n_counts} of {len(KEYS) * 3}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for j, key in enumerate([(0.5, "n2"), (0.25, "n2")]):
    c = classes(key, "r1")
    for k, col, lab in ((1, "tab:red", "excited"), (-1, "k", "inhibited"), (0, "tab:blue", "unaffected")):
        cur = np.array([cls[i][key]["curve"] for i in np.flatnonzero(c == k)])
        x = cls[0][key]["centres"]
        ax[j].plot(x, np.nanmean(cur, axis=0), color=col, label=f"{lab} (n={len(cur)})")
        print(f"{key}: {lab} mean curve range {np.nanmin(np.nanmean(cur, axis=0)):.2f} to {np.nanmax(np.nanmean(cur, axis=0)):.2f}")
    ax[j].axvspan(0, 5, color="g", alpha=0.15)
    ax[j].set(xlabel="time from light onset (s)", ylabel="normalized discharge", title=f"{key[0] * 1000:.0f} ms bins, n2, r1")
ax[0].legend()
plt.tight_layout()

**Result: not reproduced.** Under none of the twelve readings (two bin sizes, n1 and n2, r1 to r3)
are the three counts within tolerance: the method finds 36 to 70 inhibited and 68 to 118 excited
units, against 29 and 56, and the decrease of inhibited units (16.2 to 39.7%) is below 49.2 - 5.1
under every reading. Counting every light pulse, including those given with the pinch, does not
bring the counts within tolerance under any reading either. The dataset's own flags (column 17 of `data_correlation.mat`) give 56 excited,
30 inhibited and 244 unaffected units, within tolerance of the paper's counts; with those classes the
increase measured here (33.5 to 35.7%) is within tolerance and the decrease (43.1 to 46.4%) is within
it under two of four readings. The deposited classification matches the paper, while the method the
claims file gives agrees with it for 191 to 286 of 330 units.

## fig1d-excited-cells-fire-faster

> Basal forebrain units excited by the light have higher spontaneous firing rates than inhibited or
> unaffected units.

Classes from fig1-bf-response-classes under each reading. Basal rate readings from the criterion:
(a) the rate outside the light pulses over the recording, (b) the rate in the 5 s before each pulse.

In [ ]:
def basal_rates(u, a):
    L = lan[(a, u["rec"])]
    sp = u["spikes"] / 1000
    T = L["pnts"] / L["srate"]
    on, off = bf2.onsets_from_ttl(L["ttl"][:, 0], L["srate"])
    inside = sum(((sp >= x) & (sp < y)).sum() for x, y in zip(on, off))
    outside = (len(sp) - inside) / (T - (off - on).sum())
    pre = np.mean([((sp >= o - 5) & (sp < o)).sum() / 5 for o in u["rt_laser"] / 1000])
    return outside, pre


B = np.array([basal_rates(u, a) for u, a in zip(units, unit_animal)])
print(f"rate outside light: median {np.median(B[:, 0]):.2f} Hz, range {B[:, 0].min():.2f} to {B[:, 0].max():.2f}; "
      f"5 s before pulses: median {np.median(B[:, 1]):.2f}, range {B[:, 1].min():.2f} to {B[:, 1].max():.2f}")
n_ok = 0
for key in KEYS:
    for r in ("r1", "r2", "r3"):
        c = classes(key, r)
        for j, nm in enumerate(("outside light", "5 s before")):
            g = {k: B[c == k, j] for k in (1, -1, 0)}
            kw = stats.kruskal(g[1], g[-1], g[0]).pvalue
            pei = stats.ranksums(g[1], g[-1]).pvalue
            peu = stats.ranksums(g[1], g[0]).pvalue
            piu = stats.ranksums(g[-1], g[0]).pvalue
            ok = kw < 0.05 and pei < 0.05 and peu < 0.05 and np.median(g[1]) > max(np.median(g[-1]), np.median(g[0]))
            n_ok += ok
            print(f"{key[0] * 1000:.0f} ms {key[1]} {r}, {nm}: median Hz excited {np.median(g[1]):.2f}, inhibited "
                  f"{np.median(g[-1]):.2f}, unaffected {np.median(g[0]):.2f}; Kruskal-Wallis p = {kw:.2g}; rank-sum "
                  f"exc-inh p = {pei:.2g}, exc-unaff p = {peu:.2g}, inh-unaff p = {piu:.2g}; holds {ok}")
print(f"readings where the criterion holds: {n_ok} of {2 * len(KEYS) * 3}")
g = {k: B[flags == k, 0] for k in (1, -1, 0)}
print(f"variant, dataset's own classes, outside light: medians {np.median(g[1]):.2f}, {np.median(g[-1]):.2f}, "
      f"{np.median(g[0]):.2f} Hz; Kruskal-Wallis p = {stats.kruskal(g[1], g[-1], g[0]).pvalue:.2g}")
c = classes((0.5, "n2"), "r1")
for a in bf2.VP_ANIMALS:
    m = unit_animal == a
    print(f"{a} (500 ms, n2, r1, outside light): median excited {np.median(B[m & (c == 1), 0]):.2f}, "
          f"others {np.median(B[m & (c != 1), 0]):.2f} Hz (excited {(m & (c == 1)).sum()}, others {(m & (c != 1)).sum()})")

**Result: reproduced.** Under all 24 combinations of class reading and basal-rate reading, the
Kruskal-Wallis test gives p < 0.05 and excited units have a higher median rate than inhibited and
unaffected units by rank-sum tests at p < 0.05.

## fig1d-discharge-by-class

> Normalized discharge probability during the light differs between excited, inhibited and
> unaffected basal forebrain units.

In [ ]:
n_ok = 0
for key in KEYS:
    pm = np.array([c[key]["pulse_mean"] for c in cls])
    for r in ("r1", "r2", "r3"):
        c = classes(key, r)
        g = {k: pm[c == k] for k in (1, -1, 0)}
        kw = stats.kruskal(g[1], g[-1], g[0]).pvalue
        pe_u = stats.ranksums(g[1], g[0]).pvalue
        pu_i = stats.ranksums(g[0], g[-1]).pvalue
        pe_i = stats.ranksums(g[1], g[-1]).pvalue
        order = np.median(g[1]) > np.median(g[0]) > np.median(g[-1])
        ok = kw < 0.05 and max(pe_u, pu_i, pe_i) < 0.05 and order
        n_ok += ok
        print(f"{key[0] * 1000:.0f} ms {key[1]} {r}: medians excited {np.median(g[1]):.3f}, unaffected {np.median(g[0]):.3f}, "
              f"inhibited {np.median(g[-1]):.3f}; Kruskal-Wallis p = {kw:.2g}; rank-sum exc-unaff {pe_u:.2g}, "
              f"unaff-inh {pu_i:.2g}, exc-inh {pe_i:.2g}; holds {ok}")
print(f"readings where the criterion holds: {n_ok} of {len(KEYS) * 3}")

**Result: reproduced.** Under all twelve readings the Kruskal-Wallis test and the three pairwise
rank-sum tests give p < 0.05, with excited above unaffected above inhibited. As the claims file notes,
this largely follows from the classification itself.

## fig1e-excited-slower-kinetics

> Excited basal forebrain units respond with slower kinetics than inhibited ones (time constants
> 602.5 against 263.1 ms).

Per unit, a single exponential approaching a plateau, fitted to the normalized discharge from light
onset (criterion): w1 up to the first extremum of the response (the pulse bin of largest deviation
from 1), w2 over the whole pulse. The time constant is bounded to 1 ms to 20 s, a standard choice;
fits at the upper bound are counted.

In [ ]:
n_ok = 0
for key in KEYS:
    for w in ("w1", "w2"):
        tau = np.array([bf2.fit_tau(c[key]["curve"], c[key]["centres"], w) for c in cls])
        for r in ("r1", "r2", "r3"):
            c = classes(key, r)
            e, i = tau[c == 1], tau[c == -1]
            ef, if_ = e[np.isfinite(e)], i[np.isfinite(i)]
            p = stats.ranksums(ef, if_).pvalue
            ok = p < 0.05 and np.median(ef) > np.median(if_)
            n_ok += ok
            print(f"{key[0] * 1000:.0f} ms {key[1]} {r} {w}: excited mean {ef.mean():.0f} ms (median {np.median(ef):.0f}, "
                  f"{len(ef)} fitted of {len(e)}, {(ef > 19900).sum()} at the bound), inhibited mean {if_.mean():.0f} ms "
                  f"(median {np.median(if_):.0f}, {len(if_)} of {len(i)}, {(if_ > 19900).sum()} at the bound); "
                  f"rank-sum p = {p:.3g}; test holds {ok}; means within tolerance "
                  f"{within(ef.mean(), 602.5, 68.6) and within(if_.mean(), 263.1, 48.7)}")
print(f"readings where the rank-sum test holds with excited slower: {n_ok} of {len(KEYS) * 2 * 3}")

**Result: not reproduced.** Under none of the 24 readings does the rank-sum test find excited units
slower than inhibited ones at p < 0.05 (p = 0.051 to 0.992), and under none are both mean time
constants within tolerance.
Many fits run to the 20-s bound, so the means are dominated by them; the medians (printed) do not
separate the classes either.

## supp4-tail-pinch-response-patterns

> Of 44 pinch-inhibited units 10 are excited, 7 inhibited and 27 unaffected by light; of 149
> pinch-activated units 24, 11 and 114; and of 18 light-inhibited units 6 are excited, 7 inhibited
> and 5 unaffected by the pinch.

Pinch responses are classified by the same method around the per-unit pinch onsets (`RT somatic`),
with the pinch window 2 s (the files' pinch duration, Methods "1 to 2 s") or 5 s (legend), both read.
Seeds: 2 000 000 plus the unit's index.

In [ ]:
has_pinch = np.array([len(u["rt_somatic"]) > 0 for u in units])
idx = np.flatnonzero(has_pinch)
print(f"units with pinch onsets: {has_pinch.sum()}, by animal {dict(collections.Counter(unit_animal[has_pinch]))}; "
      f"pinch onsets per unit {sorted(set(len(units[i]['rt_somatic']) for i in idx))}")
sflag = np.array([u["sig_somatic"] if u["sig_somatic"] is not None else 9 for u in units])
print("dataset's own pinch flags (column 19) on units with pinch onsets:", dict(collections.Counter(sflag[has_pinch])),
      "; on units without:", dict(collections.Counter(sflag[~has_pinch])), "(9: empty)")


def tally(pc, lc):
    return (f"pinch-inhibited {(pc == -1).sum()} (light e/i/u {((pc == -1) & (lc == 1)).sum()}/{((pc == -1) & (lc == -1)).sum()}/"
            f"{((pc == -1) & (lc == 0)).sum()}), pinch-activated {(pc == 1).sum()} ({((pc == 1) & (lc == 1)).sum()}/"
            f"{((pc == 1) & (lc == -1)).sum()}/{((pc == 1) & (lc == 0)).sum()}), light-inhibited {(lc == -1).sum()} "
            f"(pinch e/i/u {((lc == -1) & (pc == 1)).sum()}/{((lc == -1) & (pc == -1)).sum()}/{((lc == -1) & (pc == 0)).sum()})")


n_groups = 0
for w in (2.0, 5.0):
    for key in KEYS:
        pin = [bf2.classify_unit(units[i]["spikes"] / 1000, units[i]["rt_somatic"] / 1000, key[0], key[1],
                                 seed=2_000_000 + i, pulse=w) for i in idx]
        for r in ("r1", "r2", "r3"):
            pc = np.array([x[r] for x in pin])
            lc = classes(key, r)[idx]
            ok = within((pc == -1).sum(), 44, 4.4) and within((pc == 1).sum(), 149, 14.9) and within((lc == -1).sum(), 18, 1.8)
            n_groups += ok
            print(f"window {w:.0f} s, {key[0] * 1000:.0f} ms {key[1]} {r}: {tally(pc, lc)}; group sizes within tolerance {ok}")
print(f"\nexpected: pinch-inhibited 44 (10/7/27), pinch-activated 149 (24/11/114), light-inhibited 18 (6/7/5)")
print(f"readings with the three group sizes within tolerance: {n_groups} of {2 * len(KEYS) * 3}")
print("variant, dataset's own flags on units with pinch onsets:", tally(sflag[has_pinch], flags[has_pinch]))
print("variant, dataset's own flags on all units:", tally(np.where(sflag == 9, 0, sflag), flags))

**Result: not reproduced.** Pinch onsets are in two animals (NE52, NE58; 242 units), while the legend
gives four animals for panels C and D. Under no reading are the three group sizes within tolerance:
pinch-activated units number at most 62 against 149, and light-inhibited units tested with the pinch
are 25 to 47 against 18 (panel E, which the legend gives for two animals). The dataset's own pinch
flags over all units, including the 88 units of NE48 and NE69 whose files hold no pinch timing, give
45 pinch-inhibited units (10, 7 and 28 by light class) against the legend's 44 (10, 7, 27), but 61
pinch-activated units against 149.

## Blocked claims

Every other claim rests on data the data set does not hold. The listings above show its contents:
basal forebrain (ventral pallidum) optrode recordings, medial septum and hippocampus recordings, and
open-field and Y-maze videos.

- supp1 (histology), supp2 and supp3 (in vitro whole-cell recordings): no histology or slice data.
- fig2c, supp6, fig2d, supp7c, fig2e, fig3a, fig3b, supp9-layers, supp9-fast-spiking-cells,
  supp9-interneurons-stronger: prefrontal unit recordings; none are in the data set (the descriptor
  names only basal forebrain, medial septum and hippocampus recordings).
- fig4bcd, supp10, fig4g, supp11a, fig4h, suppt2: prefrontal LFP; not in the data set.
- fig5c: sessions with the light path blocked; every VP open-field folder is a `Laser_ON` session.
- fig5d and supp14: the movement estimator and the laser train onsets are stored (`cambio`, `ttl`)
  for 3 of the 5 VP mice and for none of the 9 control mice, whose folders hold only an MPEG video
  and tracking output; the paper's 5 NpHR- mice cannot be told apart from the 9 controls, and the
  per-train traces in `result_rapidez_all.mat` carry no mouse or train identity and are a
  different, undocumented quantity.

## Summary

| Claim | Outcome |
|---|---|
| fig1-bf-response-classes | Not reproduced |
| fig1d-excited-cells-fire-faster | Reproduced |
| fig1d-discharge-by-class | Reproduced |
| fig1e-excited-slower-kinetics | Not reproduced |
| supp4-tail-pinch-response-patterns | Not reproduced |
| supp1, supp2, supp3 | Blocked: data missing |
| 16 prefrontal claims (Figures 2 to 4 and Supplementary Figures 6 to 11, Table 2) | Blocked: data missing |
| fig5c-light-path-needed | Blocked: data missing |
| fig5d-nphr-locomotion, supp14-moving-no-difference | Blocked: metadata insufficient |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

Differences from the paper's method: the response classification follows the readings the claims
file names, since the paper's own procedure is not given in full; the paper's analysis code is not
public. The dataset's own per-unit flags match the paper's counts, so the paper's procedure differs
from every reading run here.

What was found about the data set: recordings last 900 s (600 s in NE69), not the 10 min of the
descriptor; light pulses come every 20, 25 or 30 s, not 15 s off; the pinch TTL row of NE52 holds
hundreds of short pulses, and the pinch events are in LAN.RTb; pinch timing exists for 2 of the 4
animals; the data set holds no prefrontal data, although the paper's main findings are prefrontal;
control open-field videos carry no laser timing.

In [ ]:
print(f"total run time {(time.time() - T0) / 60:.1f} min; peak memory of this process "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")